# 🏎️ F1 BI — Análisis Exploratorio de Datos (EDA)
## Proyecto Final · Módulo 4: Inteligencia de Negocios y SQL Avanzado

**Propósito:** Explorar el dataset Ergast F1 antes del ETL para identificar:
- Cobertura y completitud de cada tabla
- Valores nulos, atípicos e inconsistencias
- Decisiones de limpieza a implementar en `transform_fact()`

| Sección | Qué analiza |
|---|---|
| 1. Setup | Instalación, imports y carga de datos |
| 2. Cobertura | Filas, columnas y % nulos por tabla |
| 3. Nulos críticos | Nulos en columnas clave de results.csv |
| 4. Pit stops | Cobertura histórica (solo desde 1994) |
| 5. Outliers | Posición de salida, delta, tiempos de vuelta |
| 6. Consistencia | IDs huérfanos entre tablas |
| 7. Distribuciones | Puntos, abandonos y carreras por era |
| 8. Decisiones ETL | Resumen de limpiezas a aplicar |

## 1. Setup — Instalación y carga de datos

In [ ]:
# Ejecutar solo la primera vez en Google Colab
!pip install kagglehub pandas matplotlib seaborn -q

In [ ]:
import os
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# Descarga del dataset desde Kaggle
DATA_PATH = kagglehub.dataset_download('rohanrao/formula-1-world-championship-1950-2020')
print('Dataset en:', DATA_PATH)

In [ ]:
# Cargar todas las tablas
def cargar_csv(nombre):
    ruta = os.path.join(DATA_PATH, nombre)
    return pd.read_csv(ruta, na_values=['\\N', 'NA', '', 'NULL']) if os.path.exists(ruta) else pd.DataFrame()

races                = cargar_csv('races.csv')
results              = cargar_csv('results.csv')
drivers              = cargar_csv('drivers.csv')
constructors         = cargar_csv('constructors.csv')
circuits             = cargar_csv('circuits.csv')
pit_stops            = cargar_csv('pit_stops.csv')
lap_times            = cargar_csv('lap_times.csv')
qualifying           = cargar_csv('qualifying.csv')
status               = cargar_csv('status.csv')
driver_standings     = cargar_csv('driver_standings.csv')
constructor_results  = cargar_csv('constructor_results.csv')

print('Tablas cargadas correctamente')

## 2. Cobertura general — Filas, columnas y % nulos por tabla

**¿Qué buscamos?** Identificar qué tablas tienen datos incompletos antes de diseñar el ETL.

In [ ]:
tablas = {
    'races': races, 'results': results, 'drivers': drivers,
    'constructors': constructors, 'circuits': circuits,
    'pit_stops': pit_stops, 'lap_times': lap_times,
    'qualifying': qualifying, 'status': status,
    'driver_standings': driver_standings, 'constructor_results': constructor_results
}

resumen = []
for nombre, df in tablas.items():
    if df.empty: continue
    pct_nulos = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100)
    cols_con_nulos = (df.isnull().sum() > 0).sum()
    resumen.append({
        'Tabla': nombre,
        'Filas': f'{len(df):,}',
        'Columnas': df.shape[1],
        'Cols con nulos': cols_con_nulos,
        '% nulos global': f'{pct_nulos:.1f}%'
    })

df_resumen = pd.DataFrame(resumen)
print(df_resumen.to_string(index=False))

## 3. Nulos críticos en `results.csv`

**¿Por qué esta tabla?** Es la fuente principal de `fact_resultado_carrera`. Cada nulo en columnas clave puede afectar medidas o relaciones del modelo dimensional.

In [ ]:
# % de nulos por columna en results
nulos_results = results.isnull().mean().mul(100).sort_values(ascending=False)
nulos_results = nulos_results[nulos_results > 0]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(nulos_results.index, nulos_results.values,
               color=['#D85A30' if v > 10 else '#378ADD' for v in nulos_results.values])
ax.set_xlabel('% de valores nulos')
ax.set_title('% de nulos por columna — results.csv', fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, nulos_results.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('\nColumnas con nulos > 10% (requieren decisión en ETL):')
print(nulos_results[nulos_results > 10].to_string())

In [ ]:
# Analizar posicion_salida = 0 (pit lane start)
grid_cero = results[results['grid'] == 0]
print(f'Registros con grid = 0 (pit lane start): {len(grid_cero):,}')
print(f'% del total: {len(grid_cero)/len(results)*100:.1f}%')
print('\nDecisión ETL: grid=0 debe convertirse a NULL (no es posición real de salida)')

# Registros donde posicion_final es nulo (no terminaron)
sin_posicion = results[results['positionOrder'].isna()]
print(f'\nRegistros sin posicion_final: {len(sin_posicion):,}')
print(f'% del total: {len(sin_posicion)/len(results)*100:.1f}%')

## 4. Cobertura histórica de `pit_stops.csv`

**Hallazgo esperado:** Los datos de pit stops solo existen desde 1994. Antes de esa fecha, `num_pitstops = 0` en la fact table no significa '0 paradas' sino 'sin datos'. Esta distinción es crítica para el análisis.

In [ ]:
if not pit_stops.empty:
    # Unir con races para obtener el año
    ps_años = pit_stops.merge(races[['raceId', 'year']], on='raceId', how='left')
    cobertura = ps_años.groupby('year').size().reset_index(name='registros')

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(cobertura['year'], cobertura['registros'],
           color=['#D85A30' if y < 1994 else '#378ADD' for y in cobertura['year']])
    ax.axvline(x=1993.5, color='red', linestyle='--', linewidth=1.5, label='Inicio cobertura (1994)')
    ax.set_xlabel('Año')
    ax.set_ylabel('Registros de pit stops')
    ax.set_title('Cobertura histórica de pit_stops.csv', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

    años_sin_datos = races[races['year'] < pit_stops.merge(
        races[['raceId','year']],on='raceId').year.min()]['year'].nunique()
    print(f'Años SIN datos de pit stops: {años_sin_datos}')
    print('Decisión ETL: agregar flag `tiene_datos_pitstop` en la fact o documentar la limitación')
else:
    print('pit_stops.csv no encontrado')

## 5. Outliers — Posición de salida, delta_posicion y tiempos de vuelta

**¿Qué buscamos?** Valores fuera de rango que podrían ser errores de captura o casos especiales que necesitan tratamiento en el ETL.

In [ ]:
# Distribución de posicion_salida (grid)
grid_valido = results[results['grid'] > 0]['grid']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histograma de posición de salida
axes[0].hist(grid_valido, bins=30, color='#378ADD', edgecolor='white')
axes[0].set_title('Distribución de posición de salida (grid > 0)', fontweight='bold')
axes[0].set_xlabel('Posición de salida')
axes[0].set_ylabel('Frecuencia')

# Boxplot de posición de salida
axes[1].boxplot(grid_valido, vert=False, patch_artist=True,
                boxprops=dict(facecolor='#E6F1FB', color='#378ADD'),
                medianprops=dict(color='#D85A30', linewidth=2))
axes[1].set_title('Boxplot — posición de salida', fontweight='bold')
axes[1].set_xlabel('Posición')
plt.tight_layout()
plt.show()

print(f'Posición máxima de salida registrada: {grid_valido.max()}')
print(f'Registros con grid > 26 (grillas grandes, pre-1980s): {(grid_valido > 26).sum():,}')

In [ ]:
# Delta posicion: posicion_salida - posicion_final
res_valido = results[(results['grid'] > 0) & results['positionOrder'].notna()].copy()
res_valido['delta'] = res_valido['grid'] - res_valido['positionOrder']

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(res_valido['delta'], bins=50, color='#5B1180', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Sin cambio de posición')
ax.set_title('Distribución de delta_posicion (salida - llegada)', fontweight='bold')
ax.set_xlabel('Posiciones ganadas (+) o perdidas (-)')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Delta máximo positivo (más posiciones ganadas): {res_valido["delta"].max()}')
print(f'Delta máximo negativo (más posiciones perdidas): {res_valido["delta"].min()}')
print(f'Mediana del delta: {res_valido["delta"].median()}')

In [ ]:
# Tiempos de vuelta: detectar outliers en lap_times
if not lap_times.empty:
    lt = lap_times[lap_times['milliseconds'].notna()].copy()
    lt['segundos'] = lt['milliseconds'] / 1000

    p01 = lt['segundos'].quantile(0.01)
    p99 = lt['segundos'].quantile(0.99)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(lt['segundos'], bins=100, color='#0F6E56', edgecolor='white', alpha=0.85)
    axes[0].set_title('Distribución de tiempos de vuelta (todos)', fontweight='bold')
    axes[0].set_xlabel('Segundos')
    axes[0].set_ylabel('Frecuencia')

    lt_limpio = lt[(lt['segundos'] >= p01) & (lt['segundos'] <= p99)]
    axes[1].hist(lt_limpio['segundos'], bins=100, color='#0F6E56', edgecolor='white', alpha=0.85)
    axes[1].set_title('Distribución de tiempos de vuelta (sin extremos p1-p99)', fontweight='bold')
    axes[1].set_xlabel('Segundos')
    plt.tight_layout()
    plt.show()

    outliers_bajos = (lt['segundos'] < p01).sum()
    outliers_altos = (lt['segundos'] > p99).sum()
    print(f'Tiempos < percentil 1 ({p01:.1f}s): {outliers_bajos:,} registros (safety car, pit entry laps)')
    print(f'Tiempos > percentil 99 ({p99:.1f}s): {outliers_altos:,} registros (vueltas muy lentas)')
    print('Decisión ETL: filtrar outliers de lap_times con rango p1-p99 para análisis de ritmo')

## 6. Consistencia entre tablas — IDs huérfanos

**¿Qué buscamos?** Verificar que todos los IDs en `results.csv` tengan correspondencia en sus tablas de referencia. Un ID huérfano rompe el JOIN en el ETL.

In [ ]:
checks = [
    ('results.driverId',      results['driverId'],      drivers['driverId'],       'drivers'),
    ('results.constructorId', results['constructorId'], constructors['constructorId'], 'constructors'),
    ('results.raceId',        results['raceId'],        races['raceId'],            'races'),
    ('results.statusId',      results['statusId'],      status['statusId'],         'status'),
    ('races.circuitId',       races['circuitId'],       circuits['circuitId'],      'circuits'),
]

print(f'{'Check':<35} {'Huérfanos':>10} {'Estado':>12}')
print('-' * 60)
for nombre, ids_fact, ids_dim, tabla_dim in checks:
    huerfanos = (~ids_fact.isin(ids_dim)).sum()
    estado = '✅ OK' if huerfanos == 0 else f'⚠️  {huerfanos} huérfanos'
    print(f'{nombre:<35} {huerfanos:>10,} {estado:>12}')

print('\nDecisión ETL: si hay huérfanos, agregar validación en transform_fact() antes del merge')

## 7. Distribuciones analíticas — Puntos, abandonos y carreras por era

**¿Qué buscamos?** Entender los datos desde la perspectiva de la pregunta analítica: rendimiento de pilotos y estrategias de carrera.

In [ ]:
# Carreras por año
carreras_año = races.groupby('year').size().reset_index(name='carreras')

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(carreras_año['year'], carreras_año['carreras'], alpha=0.3, color='#D85A30')
ax.plot(carreras_año['year'], carreras_año['carreras'], color='#D85A30', linewidth=2)

# Marcar eras
eras = [(1950,1966,'Asp. Natural'),(1967,1976,'DFV'),(1977,1988,'Turbo'),
        (1989,2005,'NA Moderno'),(2006,2013,'V8'),(2014,2024,'Híbrida')]
colores_era = ['#E6F1FB','#EAF3DE','#FAEEDA','#F3E6FB','#FCEBEB','#E1F5EE']
for (ini, fin, nombre), color in zip(eras, colores_era):
    ax.axvspan(ini, fin, alpha=0.25, color=color)
    ax.text((ini+fin)/2, ax.get_ylim()[1]*0.92, nombre, ha='center', fontsize=8, color='#444')

ax.set_title('Número de carreras por año y era técnica', fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('Carreras en la temporada')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de puntos por resultado (sistema actual vs histórico)
puntos_dist = results[results['points'] > 0]['points'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(puntos_dist.index.astype(str), puntos_dist.values, color='#378ADD', edgecolor='white')
ax.set_title('Distribución de valores de puntos otorgados (sistemas históricos)', fontweight='bold')
ax.set_xlabel('Puntos')
ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()

print('Sistemas de puntos distintos en el dataset:')
print('  1950-1959: 8-6-4-3-2 + vuelta rápida')
print('  1960-1990: 9-6-4-3-2-1')
print('  1991-2002: 10-6-4-3-2-1')
print('  2003-2009: 10-8-6-5-4-3-2-1')
print('  2010+    : 25-18-15-12-10-8-6-4-2-1')
print('\nDecisión ETL: normalizar puntos por era para comparaciones cross-temporal')

In [ ]:
# Tasa de abandono por era
results_era = results.merge(races[['raceId','year']], on='raceId', how='left')

def era_label(y):
    if y<=1966: return '1. Asp.Natural\n1950-66'
    elif y<=1976: return '2. DFV\n1967-76'
    elif y<=1988: return '3. Turbo\n1977-88'
    elif y<=2005: return '4. NA Mod.\n1989-05'
    elif y<=2013: return '5. V8\n2006-13'
    else: return '6. Hibrida\n2014+'

results_era['era'] = results_era['year'].apply(era_label)

# Categorizar abandonos
status_fin = status[status['status']=='Finished']['statusId'].values
results_era['abandono'] = ~results_era['statusId'].isin(status_fin)

tasa_era = results_era.groupby('era')['abandono'].mean().mul(100).reset_index()

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(tasa_era['era'], tasa_era['abandono'],
              color=['#D85A30' if v > 30 else '#378ADD' for v in tasa_era['abandono']])
ax.set_title('Tasa de abandono por era técnica (%)', fontweight='bold')
ax.set_ylabel('% de abandonos')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, tasa_era['abandono']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.5, f'{val:.1f}%',
            ha='center', fontsize=11, fontweight='500')
plt.tight_layout()
plt.show()

print('Hallazgo: la era turbo (1977-88) tuvo la mayor tasa de abandono mecánico de la historia')

In [ ]:
# Top 10 constructores por victorias históricas
victorias = results[results['positionOrder']==1].merge(
    constructors[['constructorId','name']], on='constructorId', how='left'
)
top_constructores = victorias['name'].value_counts().head(10).reset_index()
top_constructores.columns = ['constructor','victorias']

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.barh(top_constructores['constructor'][::-1],
               top_constructores['victorias'][::-1], color='#D85A30', edgecolor='white')
ax.set_title('Top 10 constructores por victorias históricas (1950-2024)', fontweight='bold')
ax.set_xlabel('Victorias')
for bar, val in zip(bars, top_constructores['victorias'][::-1]):
    ax.text(val+1, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Posicion salida vs posicion final: ¿qué tanto importa el grid?
res_fin = results[(results['grid']>0) & (results['positionOrder'].notna())].copy()
res_fin['grid']          = res_fin['grid'].astype(int)
res_fin['positionOrder'] = res_fin['positionOrder'].astype(int)

# Porcentaje que gana desde cada posición de salida (top 10)
res_top10 = res_fin[res_fin['grid'] <= 10]
conv = res_top10.groupby('grid').apply(
    lambda x: (x['positionOrder']==1).mean()*100
).reset_index()
conv.columns = ['grid','pct_victoria']

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(conv['grid'], conv['pct_victoria'], color='#5B1180', edgecolor='white')
ax.set_title('% de victorias según posición de salida (grids 1-10)', fontweight='bold')
ax.set_xlabel('Posición de salida')
ax.set_ylabel('% de veces que ganó la carrera')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticks(range(1,11))
plt.tight_layout()
plt.show()

polo = conv[conv['grid']==1]['pct_victoria'].values[0]
print(f'Desde la pole position se gana el {polo:.1f}% de las carreras')
print('Hallazgo clave para la pregunta analítica: la posición de salida es el factor más importante')

## 8. Resumen de decisiones ETL — Hallazgos del EDA

In [ ]:
decisiones = [
    ('grid = 0',         'pit lane start',    'Convertir a NULL en posicion_salida'),
    ('milliseconds nulo','abandono/no data',  'Mantener NULL en tiempo_total_ms, no imputar'),
    ('pit_stops < 1994', 'sin cobertura',     'num_pitstops=0 antes de 1994 significa sin datos, no 0 paradas'),
    ('lap_times extremos','safety car/lento', 'Filtrar p1-p99 en análisis de ritmo, no en la fact'),
    ('puntos multisistema','5 sistemas hist.', 'Guardar puntos reales; calcular puntos normalizados en SQL'),
    ('IDs huérfanos',    'ver celda 6',       'Validar con assert antes de los merges en transform_fact'),
]

print(f"{'Problema encontrado':<25} {'Causa':<22} {'Decisión en ETL'}")
print('=' * 85)
for prob, causa, decision in decisiones:
    print(f'{prob:<25} {causa:<22} {decision}')

print('\nEstas decisiones ya están implementadas en etl_pipeline.ipynb.')
print('El EDA las justifica con evidencia real del dataset.')

## 8. Resumen del EDA — Secciones y decisiones de limpieza para el ETL

### 📋 El notebook tiene 8 secciones con análisis progresivos

| # | Sección | Qué analiza |
|:---:|---|---|
| 1 | **Setup** | Instalación, imports, credenciales Kaggle y carga de los 14 CSVs |
| 2 | **Cobertura** | Filas, columnas y % de nulos global por cada tabla del dataset |
| 3 | **Nulos críticos** | Gráfica de % nulos por columna en `results.csv`; detecta `grid=0` como pit lane start |
| 4 | **Pit stops** | Cobertura histórica: datos solo existen desde 1994 |
| 5 | **Outliers** | Histogramas y boxplot de `grid`, `delta_posicion` y tiempos de vuelta (p1–p99) |
| 6 | **Consistencia** | Tabla de IDs huérfanos entre todas las tablas relacionales |
| 7 | **Distribuciones** | Puntos por era, tasa de abandono y % de victorias por posición de salida |
| 8 | **Decisiones ETL** | Resumen que conecta cada hallazgo del EDA con una acción concreta en el pipeline |


In [ ]:
from IPython.display import display, HTML

# ── Tabla 1: Secciones del notebook ─────────────────────────────────────────
secciones_html = """
<style>
  .f1-table { border-collapse: collapse; width: 100%; font-family: 'Segoe UI', sans-serif; font-size: 13px; margin-bottom: 28px; }
  .f1-table thead tr { background: #1A1A1A; color: #fff; }
  .f1-table thead th { padding: 9px 14px; text-align: left; font-size: 11px; letter-spacing: 0.08em; text-transform: uppercase; }
  .f1-table thead th:first-child { border-left: 4px solid #E8231A; }
  .f1-table tbody tr { border-bottom: 1px solid #E0DED8; }
  .f1-table tbody tr:hover { background: #FAF9F6; }
  .f1-table tbody td { padding: 8px 14px; vertical-align: top; }
  .f1-table tbody td:first-child { border-left: 4px solid transparent; font-weight: 600; color: #1A1A1A; text-align: center; width: 40px; }
  .f1-table tbody tr:hover td:first-child { border-left-color: #E8231A; }
  .badge { display: inline-block; font-size: 11px; font-weight: 600; padding: 2px 8px; border-radius: 20px; background: #F0EFEB; color: #666460; }
  code { font-size: 11.5px; background: #F0EFEB; padding: 1px 5px; border-radius: 3px; color: #E8231A; font-family: monospace; }
  .footer-note { background: #FFFBF0; border: 1px solid #E0DED8; border-left: 4px solid #F5A623;
                 border-radius: 5px; padding: 10px 14px; font-size: 12.5px; color: #555; line-height: 1.6; margin-top: 4px; }
  .section-title { font-family: 'Segoe UI', sans-serif; font-size: 12px; font-weight: 700;
                   letter-spacing: 0.1em; text-transform: uppercase; color: #888; margin: 0 0 8px; }
</style>

<p class="section-title">🏎 El notebook tiene 8 secciones con análisis progresivos</p>
<table class="f1-table">
  <thead><tr><th>#</th><th>Sección</th><th>Qué analiza</th></tr></thead>
  <tbody>
    <tr><td>1</td><td><strong>Setup</strong></td><td>Instalación, imports, credenciales Kaggle y carga de los 14 CSVs</td></tr>
    <tr><td>2</td><td><strong>Cobertura</strong></td><td>Filas, columnas y % de nulos global por cada tabla del dataset</td></tr>
    <tr><td>3</td><td><strong>Nulos críticos</strong></td><td>Gráfica de % nulos por columna en <code>results.csv</code>; detecta <code>grid=0</code> como pit lane start</td></tr>
    <tr><td>4</td><td><strong>Pit stops</strong></td><td>Cobertura histórica: datos solo existen desde 1994</td></tr>
    <tr><td>5</td><td><strong>Outliers</strong></td><td>Histogramas y boxplot de <code>grid</code>, <code>delta_posicion</code> y tiempos de vuelta (p1–p99)</td></tr>
    <tr><td>6</td><td><strong>Consistencia</strong></td><td>Tabla de IDs huérfanos entre todas las tablas relacionales</td></tr>
    <tr><td>7</td><td><strong>Distribuciones</strong></td><td>Puntos por era, tasa de abandono y % de victorias por posición de salida</td></tr>
    <tr><td>8</td><td><strong>Decisiones ETL</strong></td><td>Resumen que conecta cada hallazgo con una acción concreta en el pipeline</td></tr>
  </tbody>
</table>
"""

# ── Tabla 2: Hallazgos → Decisiones ETL ─────────────────────────────────────
decisiones_html = """
<p class="section-title">🔧 Hallazgos del EDA → Decisiones de limpieza en el ETL</p>
<table class="f1-table">
  <thead>
    <tr><th>Problema encontrado</th><th>Causa</th><th>Decisión en ETL</th></tr>
  </thead>
  <tbody>
    <tr>
      <td>grid = 0</td>
      <td><span class="badge">pit lane start</span></td>
      <td>Convertir a <code>NULL</code> en <code>posicion_salida</code></td>
    </tr>
    <tr>
      <td>milliseconds nulo</td>
      <td><span class="badge">abandono / no data</span></td>
      <td>Mantener <code>NULL</code> en <code>tiempo_total_ms</code>, no imputar</td>
    </tr>
    <tr>
      <td>pit_stops &lt; 1994</td>
      <td><span class="badge">sin cobertura</span></td>
      <td><code>num_pitstops = 0</code> antes de 1994 significa sin datos, no 0 paradas. Documentar limitación.</td>
    </tr>
    <tr>
      <td>lap_times extremos</td>
      <td><span class="badge">safety car / lento</span></td>
      <td>Filtrar percentiles p1–p99 en análisis de ritmo, no en la fact table</td>
    </tr>
    <tr>
      <td>puntos multisistema</td>
      <td><span class="badge">5 sistemas hist.</span></td>
      <td>Guardar puntos reales; calcular puntos normalizados por era en SQL avanzado</td>
    </tr>
    <tr>
      <td>IDs huérfanos</td>
      <td><span class="badge">ver celda 6</span></td>
      <td>Validar con <code>assert</code> antes de los merges en <code>transform_fact()</code></td>
    </tr>
  </tbody>
</table>

<div class="footer-note">
  <strong>ℹ Nota:</strong> Estas decisiones están implementadas en <code>etl_pipeline.ipynb</code>.
  El EDA las justifica con evidencia real del dataset — cada limpieza tiene una causa documentada,
  no es limpieza genérica a ciegas.
</div>
"""

display(HTML(secciones_html + decisiones_html))